In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [5]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_22_8_7,0.999458,0.951864,0.999575,0.997981,0.999136,0.000538,0.047724,0.000519,0.001260,0.000890,0.023589,0.023185,1.000334,0.024172,141.056924,217.846101,"Hidden Size=[7, 3], regularizer=0.5, learning_..."
1,model_22_8_8,0.999457,0.951711,0.999538,0.997800,0.999059,0.000538,0.047875,0.000564,0.001373,0.000969,0.023360,0.023199,1.000334,0.024187,141.054546,217.843723,"Hidden Size=[7, 3], regularizer=0.5, learning_..."
2,model_22_8_6,0.999454,0.952020,0.999612,0.998147,0.999208,0.000541,0.047569,0.000474,0.001156,0.000815,0.023837,0.023259,1.000336,0.024250,141.044181,217.833358,"Hidden Size=[7, 3], regularizer=0.5, learning_..."
3,model_22_8_9,0.999453,0.951562,0.999501,0.997609,0.998979,0.000542,0.048023,0.000610,0.001492,0.001051,0.023138,0.023288,1.000337,0.024280,141.039233,217.828410,"Hidden Size=[7, 3], regularizer=0.5, learning_..."
4,model_22_8_5,0.999446,0.952178,0.999649,0.998297,0.999275,0.000549,0.047413,0.000429,0.001063,0.000746,0.024110,0.023433,1.000341,0.024431,141.014390,217.803567,"Hidden Size=[7, 3], regularizer=0.5, learning_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,model_18_8_2,0.996454,0.955418,0.998805,0.999857,0.999515,0.003516,0.044200,0.000491,0.000122,0.000307,0.043407,0.059294,1.002300,0.061818,133.301014,207.652439,"Hidden Size=[5, 5], regularizer=0.5, learning_..."
1023,model_14_6_7,0.996351,0.897023,0.996008,0.999145,0.998918,0.003618,0.102094,0.001080,0.000381,0.000731,0.080047,0.060148,1.002246,0.062709,137.243796,214.032973,"Hidden Size=[6, 4], regularizer=0.5, learning_..."
1116,model_6_9_1,0.996146,0.959867,0.999865,0.999893,0.999876,0.003821,0.039789,0.000130,0.000065,0.000097,0.042974,0.061818,1.001682,0.064449,169.134273,265.425463,"Hidden Size=[6, 6], regularizer=0.5, learning_..."
1123,model_18_8_1,0.996060,0.955009,0.998820,0.999764,0.999458,0.003906,0.044605,0.000484,0.000201,0.000343,0.043855,0.062497,1.002555,0.065158,133.090535,207.441961,"Hidden Size=[5, 5], regularizer=0.5, learning_..."
